# Demo: PyApprox integration with SPAROW.

## Discrete Facility Location Models

This demonstrates how to use the sample allocation scheme recommended by PyApprox in the UQ workflow of SPAROW. 

This helps allocate finite computational budget between high-fidelity model evaluations and low-fidelity model evaluations, such that we achieve maximum variance reduction for the optimality gap estimator within specified computational budget.

This tutorial was adapted from the PyApprox docs: https://sandialabs.github.io/pyapprox/multifidelity_estimation_cookbook.html 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pyapprox.util.backends.numpy import NumpyBkd
from pyapprox.statest.statistics import MultiOutputMean
from pyapprox.statest.mc_estimator import MCEstimator
from pyapprox.statest import MFMCEstimator
from pyapprox.statest.acv import default_allocator_factory
from pyapprox.statest.acv.base import FittedACVEstimator

from pyapprox.statest.allocation import MCAllocator

from pyapprox.optimization.minimize.scipy.slsqp import ScipySLSQPOptimizer

from sparow.conf_intervals.options import UQOptions
from sparow.conf_intervals.acv_mrp import ACVMRP
from sparow.conf_intervals.evaluate_true_optimality_gap import TrueOptimalityGapEvaluator
from sparow.conf_intervals.pyapprox_interface import (
    convert_pyapprox_allocation_to_acvmrp_params,
    build_pyapprox_mf_problem_from_ensemble,
)


[    0.00] Initializing mpi-sppy
Alternative solutions package from or_topas is available.


In [2]:
# ------------------------------------------------------
# User settings
# ------------------------------------------------------

# This is the function you have to define for your problem instance
from sparow_examples.mrp_facilityloc.mrp_discrete_facilityloc import get_model_ensemble_for_uq

MODEL_NAME = "HF" #HF and LF models evaluated on same scenario batches

# Every replication makes use of BATCH_SIZE iid draws of scenarios
BATCH_SIZE = 100
SOLVER_NAME = "gurobi_direct"
SEED = 678

# Fixed candidate solution for testing
xhat = {
    "x[0]": 0.0,
    "x[1]": 0.0,
    "x[2]": 0.0,
    "x[3]": 0.0,
    "x[4]": 1.0,
    "x[5]": 1.0,
}

In [3]:
# ------------------------------------------------------
# Build HF/LF ensemble
# ------------------------------------------------------
ensemble = get_model_ensemble_for_uq(
    model_name=MODEL_NAME,
    use_integer=False,
    seed=SEED,
    with_replacement=True,
    lf_model_type="",
) # this contains SPModelWrapperforUQ objects


In [4]:
# ------------------------------------------------------
# Build PyApprox multifidelity problem
# ------------------------------------------------------

# SLSQP is more robust than the default trust-constr optimizer for ACV allocation
optimizer = ScipySLSQPOptimizer(maxiter=200)
allocator_factory = lambda est: default_allocator_factory(est, optimizer=optimizer)

problem, bkd = build_pyapprox_mf_problem_from_ensemble(
    ensemble=ensemble,
    xhat=xhat,
    batch_size=BATCH_SIZE,
    solver_name=SOLVER_NAME,
    solver_options=None,
    seed=SEED,
    hf_cost_delay_seconds=1.0, # HACK: Artificially inflate the costs for checking code correctness
    lf_cost_delay_seconds=0.0,
)

# In this construction:
#   model 0 = HF replication-level gap estimator F_n(\hat{x})
#   model 1 = LF replication-level gap estimator G_n(\hat{x})
# and the PyApprox prior is the empirical distribution of full batches of scenarios.

models = problem.models() # this contains PyApproxModelWrapper objects
variable = problem.prior()
costs = problem.costs()
nmodels = len(models)
nqoi = models[0].nqoi()

print("Model wrapper types:")
for idx, model in enumerate(models):
    print(idx, type(model), callable(model))

# This prints the estimated wall-clock cost of one replication-level
# evaluation of each model. Model 0 is HF, model 1 is LF.
# These costs are the inputs to PyApprox's sample allocation step.
costs_np = bkd.to_numpy(costs)
for a, c in enumerate(costs_np):
    print(f"  model {a}: estimated cost = {c:.3f}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF

Model wrapper types:
0 <class 'sparow.conf_intervals.pyapprox_interface.PyApproxModelWrapper'> True
1 <class 'sparow.conf_intervals.pyapprox_interface.PyApproxModelWrapper'> True
  model 0: estimated cost = 3.176
  model 1: estimated cost = 1.961


## Stage 1: Pilot Study

Evaluate all models at a shared set of pilot samples to estimate the cross-covariance.

In [5]:
np.random.seed(42)

N_pilot = 10

# Draw N_pilot independent scenario batches from the prior.
# Each batch is one replication in the MRP / ACV-MRP sense.
samples_pilot = variable.rvs(N_pilot)

# Evaluate every model on the same pilot batches.
vals_pilot = [m(samples_pilot) for m in models]

stat = MultiOutputMean(nqoi, bkd)
cov_pilot, = stat.compute_pilot_quantities(vals_pilot)
stat.set_pilot_quantities(cov_pilot)

# Inspect pilot correlations with HF model
# In the 2-model setting, this is the empirical analogue of \rho_{fg}.
cov_np = bkd.to_numpy(cov_pilot)
print("Pilot covariance matrix:")
print(cov_np)
print("\n")

for a in range(1, nmodels):
    rho = cov_np[0, a] / np.sqrt(cov_np[0, 0] * cov_np[a, a])
    print(f"  Pilot correlation ρ(f0, f{a}) = {rho:.4f}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF

Pilot covariance matrix:
[[1.96736538e+10 1.81603834e+10]
 [1.81603834e+10 1.70811303e+10]]


  Pilot correlation ρ(f0, f1) = 0.9907


In [6]:
# ------------------------------------------------------
# PyApprox allocation
# ------------------------------------------------------

total_budget = 100.0 # Total computational budget is in terms of wall clock time
pilot_cost   = float(costs_np.sum()) * N_pilot
remaining    = total_budget - pilot_cost
print(f"Pilot cost: {pilot_cost}")
print(f"Remaining budget: {remaining}")

Pilot cost: 51.37474596500397
Remaining budget: 48.62525403499603


## Stage 2: Build Estimator and Allocate

In [7]:
est = MFMCEstimator(stat, costs)
allocator = default_allocator_factory(est)
result = allocator.allocate(remaining)
fitted = FittedACVEstimator(est, result)

# This prints the PyApprox-recommended number of replication-level evaluations
# for each model under the remaining budget.
print(f"PyApprox Samples per model (HF total, LF total): {fitted.nsamples_per_model()}")

m, M = convert_pyapprox_allocation_to_acvmrp_params(fitted.nsamples_per_model())
print(f"Translated ACV-MRP counts:")
print(f"Number of paired replications: {m}")
print(f"Number of additional LF replications: {M}")

# This is PyApprox's predicted standard deviation of the final multifidelity
# based on the pilot covariance estimates and the chosen sample allocation.
print(f"Predicted PyApprox std: {float(fitted.covariance()[0,0])**0.5:.3f}")

PyApprox Samples per model (HF total, LF total): [ 2 20]
Translated ACV-MRP counts:
Number of paired replications: 2
Number of additional LF replications: 18
Predicted PyApprox std: 33886.640


## Stage 3: Generate Samples

In [8]:
# This shows the shapes of the actual sampled inputs allocated to each model.
# Each column corresponds to one scenario batch / one replication.
samples_per_model = fitted.generate_samples_per_model(variable.rvs)
print(f"Sample shapes: {[s.shape for s in samples_per_model]}")

Sample shapes: [(400, 2), (400, 20)]


## Stage 4: Evaluate Models

In [9]:
# Evaluate each model on its allocated batches
values_per_model = [models[a](samples_per_model[a]) for a in range(nmodels)]

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF

## Stage 5: Compute Estimate

In [10]:
# PyApprox estimated optimality gap
estimate = fitted(values_per_model)
estimate_scalar = np.asarray(estimate).item()
print("PyApprox point estimator:")
print(f"Estimated mean of HF replication outputs, E[F_n(xhat)]: {estimate_scalar:.3f}")

PyApprox point estimator:
Estimated mean of HF replication outputs, E[F_n(xhat)]: 6843743.319


In [11]:
# ------------------------------------------------------
# Run ACV-MRP with translated (m, M)
# ------------------------------------------------------
options = UQOptions(
    n=BATCH_SIZE,
    m=m,
    M=M,
    alpha=0.05,
    seed=SEED,
    with_replacement=True,
    solver_name=SOLVER_NAME,
    verbose=True,
)

acv = ACVMRP(
    hf_model=ensemble.high_fidelity_model(),
    lf_model=ensemble.low_fidelity_model(),
    options=options,
)

results = acv.run(xhat=xhat)

print("\nACV-MRP results:")
print(f"ACV-MRP Point estimate: {results['point_estimate']}")
print(f"HF-only point estimate: {results['point_estimate_hf_only']}")
print(f"CI: [{results['ci_lower']}, {results['ci_upper']}]")
print(f"Estimated control variate coefficient: {results['control_variate_coefficient']}")
print(f"Estimated sample correlation: {results['sample_correlation']}")


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Running ACV-MRP with m=2, M=18, n=100
Using precomputed superset of scenarios for nested sampling scheme: False
Running paired ACV-MRP replication 1/2


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for high-fidelity paired replication 1: F_nk = 6737236.0
Gap estimate for low-fidelity paired replication 1 : G_nk = 6152332.5
Running paired ACV-MRP replication 2/2


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


...

Gap estimate for low-fidelity additional replication 18 : G_nk = 6152622.5
Running LF-only replication 17/18


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for low-fidelity additional replication 19 : G_nk = 6201222.5
Running LF-only replication 18/18


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP


Gap estimate for low-fidelity additional replication 20 : G_nk = 6188095.0

ACV-MRP results:
ACV-MRP Point estimate: 6838926.655420252
HF-only point estimate: 6914059.749999999
CI: [0.0, 6930901.243920862]
Estimated control variate coefficient: 1.1262390229532675
Estimated sample correlation: 1.0000000000000002


In [12]:
# ------------------------------------------------------
# True finite-population HF gap
# ------------------------------------------------------
true_gap_evaluator = TrueOptimalityGapEvaluator(
    model=ensemble.high_fidelity_model(),
    solver_name=SOLVER_NAME,
    solver_options=None,
)

true_gap_results = true_gap_evaluator.compute_true_gap(xhat=xhat)

print("\nTrue finite-population HF quantities:")
print(f"True optimal value: {true_gap_results['true_optimal_value']}")
print(f"xhat true value: {true_gap_results['xhat_true_value']}")
print(f"True optimality gap: {true_gap_results['true_gap']}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP



True finite-population HF quantities:
True optimal value: 9606599.999999875
xhat true value: 16475151.599999784
True optimality gap: 6868551.599999908


## Compare againt MC

In [13]:
stat_mc = MultiOutputMean(nqoi, bkd)
stat_mc.set_pilot_quantities(cov_pilot[:1, :1])
mc_est = MCEstimator(stat_mc, costs[:1]) # This is HF-only MC Estimator
mc_fitted = MCAllocator(mc_est).allocate(remaining)

mc_var  = float(mc_fitted.covariance()[0, 0])
mf_var  = float(fitted.covariance()[0, 0])
print(f"\nHF-only MC std (same computational budget): {mc_var**0.5}")
print(f"PyApprox MF std (same computational budget): {mf_var**0.5}")
print(f"Variance reduction: {mc_var / mf_var:.2f}×")


HF-only MC std (same computational budget): 36215.69991592044
PyApprox MF std (same computational budget): 33886.639840052216
Variance reduction: 1.14×
